In [3]:
import sys
print(sys.executable)

import subprocess
import os

BASE_DIR = os.getcwd()
INPUT_DIRECTORY = os.path.join(BASE_DIR, "images", "original")
OUTPUT_BASE_DIRECTORY = os.path.join(BASE_DIR,  "images", "processed")
OUTPUT_MPI_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "mpi")
OUTPUT_OMP_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "omp")
OUTPUT_HYB_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "hyb")
OUTPUT_CUDA_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, "cuda")

print(INPUT_DIRECTORY)
print(os.listdir(INPUT_DIRECTORY))

import re

from PIL import Image

/usr/local/Anaconda3-2025.06/bin/python
/users/eleves-a/2024/toan.lopez/parallel-gif-filter/images/original
['051009.vince.gif', '1.gif', '200_s.gif', '9815573.gif', 'Campusplan-Hausnr.gif', 'Campusplan-Mobilitaetsbeschraenkte.gif', 'Mandelbrot-large.gif', 'Produits_sous_linux.gif', 'TimelyHugeGnu.gif', 'australian-flag-large.gif', 'fire.gif', 'frame_002.gif', 'giphy-3.gif', 'porsche.gif', 'tumblr_myxlbtwVEb1qzt4vjo1_r14_500_large.gif', 'walla.gif', 'wallace.gif', 'walle.gif']


In [4]:
def run_command(cmd, env=None):
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env, shell=True)
    return result.stdout.decode()

# MPI
def run_sobelf_mpi(input_file, output_file, ranks=16, nodes=1):
    input_path = os.path.join(INPUT_DIRECTORY, input_file)
    output_path = os.path.join(OUTPUT_MPI_DIRECTORY, output_file)
    
    cmd = f"salloc -N {nodes} -n {ranks} mpirun ./sobelf_mpi {input_path} {output_path}"
    return run_command(cmd)

# OpenMP
def run_sobelf_omp(input_file, output_file, threads=16):
    input_path = os.path.join(INPUT_DIRECTORY, input_file)
    output_path = os.path.join(OUTPUT_OMP_DIRECTORY, output_file)

    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = str(threads)
    cmd = f"salloc -n 1 ./sobelf_omp {input_path} {output_path}"
    return run_command(cmd, env=env)

# OpenMP + MPI
def run_sobelf_hyb(input_file, output_file, ranks=4, threads=4, nodes=1):
    input_path = os.path.join(INPUT_DIRECTORY, input_file)
    output_path = os.path.join(OUTPUT_HYB_DIRECTORY, output_file)

    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = str(threads)
    cmd = f"salloc -N {nodes} -n {ranks} ./sobelf_hyb {input_path} {output_path}"
    return run_command(cmd, env=env)

# CUDA
def run_sobelf_cuda(input_file, output_file):
    input_path = os.path.join(INPUT_DIRECTORY, input_file)
    output_path = os.path.join(OUTPUT_CUDA_DIRECTORY, output_file)
    cmd = f"salloc -N 1 -n 1 ./sobelf_cuda {input_path} {output_path}"
    return run_command(cmd)



In [5]:

# Parse output
def parse_sobel_filter_output(output):
    match = re.search(r"SOBEL done in ([0-9.]+) s", output)
    if match:
        return float(match.group(1))
    else:
        raise ValueError("couldn't parse output :\n" + output + '\n')

def profile_mpi(input_file, output_file, ranks=16, nodes=1):
    return parse_sobel_filter_output(run_sobelf_mpi(input_file, output_file, ranks, nodes))

def profile_omp(input_file, output_file, threads=16):
    return parse_sobel_filter_output(run_sobelf_omp(input_file, output_file, threads))

def profile_hyb(input_file, output_file, ranks=4, threads=4, nodes=1):
    return parse_sobel_filter_output(run_sobelf_hyb(input_file, output_file, ranks, threads, nodes))

def profile_cuda(input_file, output_file):
    return parse_sobel_filter_output(run_sobelf_mpi(input_file, output_file))


In [6]:
def get_gif_info(path):
    sizes = []
    with Image.open(os.path.join(INPUT_DIRECTORY, path)) as im:
        num_frames = im.n_frames
        for frame_index in range(num_frames):
            im.seek(frame_index)
            sizes.append(im.size)
    return num_frames, sizes

In [8]:
print(profile_omp("1.gif", "1-sobel.gif"))
print(get_gif_info("1.gif"))

0.018041
(10, [(500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500), (500, 500)])


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

def create_data_set(input_dir, ranks, threads, nodes, cudas):
    # X : num_frames, width, height, ranks, threads, nodes, cuda_available
    # y : {0 : MPI , 1 : OMP , 2 : HYB , 3 : CUDA}
    pass

def build_tree(X, y, max_depth):
    tree = DecisionTreeClassifier(max_depth=max_depth)
    tree.fit(X,y)
    return tree

def print_tree(tree):
    pass

ranks = [1,2,4,8,16]
threads = [2,4,8,16]
nodes = [1,2]
cudas = [True, False]

max_depth = 5 # TODO : test

X, y = create_data_set(INPUT_DIRECTORY, ranks, threads, nodes, cudas)

tree = build_tree(X, y, 5)

print_tree(tree)
